In [1]:
from pathlib import Path
import re
import shutil
import pandas as pd

final_dir = Path("/home/jupyter/TFM_ProcesadoDeAudios/data/diarization_outputs/final_relabel")
output_dir = Path("/home/jupyter/TFM_ProcesadoDeAudios/data/diarization_outputs/consolidated")
output_dir.mkdir(parents=True, exist_ok=True)

EXPECTED_AUDIOS = 1200

def normalize_audio_id(x):
    """
    Normaliza nombres de audio/CSV para que:
    raw_915..._clean.wav
    915..._clean.wav
    raw_bajas_915..._final_merged.csv
    915..._final_merged.csv

    se interpreten como el mismo audio base.
    """
    x = Path(str(x)).name

    # quitar extensión y sufijos de CSV final
    x = re.sub(r"\.csv$", "", x)
    x = re.sub(r"_final_merged$", "", x)

    # quitar extensión y sufijos de audio
    x = re.sub(r"\.wav$", "", x)
    x = re.sub(r"_clean$", "", x)

    # quitar prefijos de ejecuciones/corpus
    prefixes = ["raw_bajas_", "raw_comercial_", "raw_", "bajas_", "comercial_"]
    changed = True
    while changed:
        changed = False
        for prefix in prefixes:
            if x.startswith(prefix):
                x = x[len(prefix):]
                changed = True

    return x

In [2]:
final_merged_files = sorted(final_dir.glob("*_final_merged.csv"))

audit_files = pd.DataFrame({
    "source_csv": [p.name for p in final_merged_files],
    "path": [str(p) for p in final_merged_files],
    "mtime": [p.stat().st_mtime for p in final_merged_files]
})

audit_files["audio_id_base"] = audit_files["source_csv"].apply(normalize_audio_id)

print("CSV *_final_merged encontrados:", audit_files["source_csv"].nunique())
print("Audio_id_base únicos normalizados:", audit_files["audio_id_base"].nunique())

dupes_files = (
    audit_files
    .groupby("audio_id_base")
    .agg(
        n_csv=("source_csv", "nunique"),
        ejemplos=("source_csv", lambda s: " | ".join(sorted(s.astype(str).head(10))))
    )
    .reset_index()
    .query("n_csv > 1")
    .sort_values("n_csv", ascending=False)
)

print("Audio_id_base con más de un CSV:", len(dupes_files))

if len(dupes_files) > 0:
    display(dupes_files.head(50))

CSV *_final_merged encontrados: 1181
Audio_id_base únicos normalizados: 1181
Audio_id_base con más de un CSV: 0


In [3]:
# Marcamos como menos preferibles los archivos con prefijos temporales
audit_files["has_temp_prefix"] = audit_files["source_csv"].str.startswith(
    ("raw_", "raw_bajas_", "raw_comercial_")
)

# Criterio:
# 1. preferir archivos sin prefijo temporal;
# 2. si hay empate, quedarse con el más reciente.
canonical_files = (
    audit_files
    .sort_values(
        ["audio_id_base", "has_temp_prefix", "mtime"],
        ascending=[True, True, False]
    )
    .drop_duplicates("audio_id_base", keep="first")
    .reset_index(drop=True)
)

dropped_files = audit_files[~audit_files["source_csv"].isin(canonical_files["source_csv"])].copy()

print("CSV originales:", len(audit_files))
print("CSV seleccionados:", len(canonical_files))
print("CSV descartados por duplicidad de audio_id_base:", len(dropped_files))
print("Audios reales normalizados seleccionados:", canonical_files["audio_id_base"].nunique())

if canonical_files["audio_id_base"].nunique() != EXPECTED_AUDIOS:
    print("⚠️ OJO: el número de audios normalizados no coincide con", EXPECTED_AUDIOS)

if len(dropped_files) > 0:
    display(dropped_files[["source_csv", "audio_id_base", "has_temp_prefix"]].head(50))

CSV originales: 1181
CSV seleccionados: 1181
CSV descartados por duplicidad de audio_id_base: 0
Audios reales normalizados seleccionados: 1181
⚠️ OJO: el número de audios normalizados no coincide con 1200


In [4]:
dfs = []

for file_path in canonical_files["path"]:
    file_path = Path(file_path)
    df_tmp = pd.read_csv(file_path)
    df_tmp["source_csv"] = file_path.name
    df_tmp["audio_id_base"] = df_tmp["audio_file"].apply(normalize_audio_id)
    dfs.append(df_tmp)

df_segments = pd.concat(dfs, ignore_index=True)

# Nos quedamos con segmentos válidos si existe la columna
if "valid_export" in df_segments.columns:
    df_segments = df_segments[df_segments["valid_export"] == True].copy()

# Recalculamos audio_id_base por seguridad
df_segments["audio_id_base"] = df_segments["audio_file"].apply(normalize_audio_id)

# Ordenamos
df_segments = df_segments.sort_values(
    ["audio_id_base", "start", "end"]
).reset_index(drop=True)

print("Dimensiones:", df_segments.shape)
print("audio_file únicos:", df_segments["audio_file"].nunique())
print("audio_id_base únicos:", df_segments["audio_id_base"].nunique())
print("Segmentos totales:", len(df_segments))
print("Columnas:", df_segments.columns.tolist())

display(df_segments.head())

Dimensiones: (40352, 26)
audio_file únicos: 1181
audio_id_base únicos: 1181
Segmentos totales: 40352
Columnas: ['segment_id_raw', 'audio_file', 'audio_stem', 'start', 'end', 'duration', 'speaker', 'rms_dbfs', 'overlap_duration_sec', 'overlap_ratio', 'valid_export', 'valid_anchor', 'drop_reasons', 'anchor_reasons', 'speaker_original', 'speaker_final', 'relabel_source', 'best_distance', 'second_distance', 'distance_margin', 'dist_SPEAKER_00', 'dist_SPEAKER_01', 'merged_n_segments', 'segment_ids_raw', 'source_csv', 'audio_id_base']


,segment_id_raw,audio_file,audio_stem,start,end,duration,speaker,rms_dbfs,overlap_duration_sec,overlap_ratio,...,relabel_source,best_distance,second_distance,distance_margin,dist_SPEAKER_00,dist_SPEAKER_01,merged_n_segments,segment_ids_raw,source_csv,audio_id_base
0,1,raw_9154117451310006851_clean.wav,raw_9154117451310006851_clean,0.030969,4.452219,4.421250,SPEAKER_01,-28.919975,0.000000,0.000000,...,centroid,0.145629,0.747864,0.602235,0.747864,0.145629,1,1,raw_9154117451310006851_clean_final_merged.csv,9154117451310006851
1,2,raw_9154117451310006851_clean.wav,raw_9154117451310006851_clean,5.211594,6.342219,1.130625,SPEAKER_00,-33.883114,0.000000,0.000000,...,centroid,0.607482,0.814511,0.207029,0.607482,0.814511,1,2,raw_9154117451310006851_clean_final_merged.csv,9154117451310006851
2,3,raw_9154117451310006851_clean.wav,raw_9154117451310006851_clean,6.426594,13.800969,7.374375,SPEAKER_01,-29.186810,0.000000,0.000000,...,centroid,0.071406,0.732181,0.660775,0.732181,0.071406,1,3,raw_9154117451310006851_clean_final_merged.csv,9154117451310006851
3,4,raw_9154117451310006851_clean.wav,raw_9154117451310006851_clean,13.800969,14.965344,1.164375,SPEAKER_00,-36.389011,0.118125,0.101449,...,centroid,0.689627,0.975324,0.285697,0.689627,0.975324,1,4,raw_9154117451310006851_clean_final_merged.csv,9154117451310006851
4,5,raw_9154117451310006851_clean.wav,raw_9154117451310006851_clean,15.285969,19.994094,4.708125,SPEAKER_01,-27.753717,0.000000,0.000000,...,centroid,0.203869,0.709531,0.505662,0.709531,0.203869,1,5,raw_9154117451310006851_clean_final_merged.csv,9154117451310006851


In [5]:
audit_segments = (
    df_segments[["audio_file", "audio_id_base", "source_csv"]]
    .drop_duplicates()
    .groupby("audio_id_base")
    .agg(
        n_audio_files=("audio_file", "nunique"),
        n_source_csv=("source_csv", "nunique"),
        audio_files=("audio_file", lambda s: " | ".join(sorted(s.astype(str).head(10)))),
        source_csvs=("source_csv", lambda s: " | ".join(sorted(s.astype(str).head(10))))
    )
    .reset_index()
    .query("n_audio_files > 1 or n_source_csv > 1")
    .sort_values(["n_source_csv", "n_audio_files"], ascending=False)
)

print("Audios normalizados en consolidado:", df_segments["audio_id_base"].nunique())
print("Audio_id_base con múltiples audio_file/source_csv después de consolidar:", len(audit_segments))

if len(audit_segments) > 0:
    display(audit_segments.head(50))
else:
    print("✅ Consolidado sin duplicidades aparentes por audio_id_base.")

Audios normalizados en consolidado: 1181
Audio_id_base con múltiples audio_file/source_csv después de consolidar: 0
✅ Consolidado sin duplicidades aparentes por audio_id_base.


In [6]:
output_path = output_dir / "all_final_merged_segments.csv"
dedup_output_path = output_dir / "all_final_merged_segments_dedup.csv"
audit_path = output_dir / "audit_final_merged_files.csv"
dropped_path = output_dir / "dropped_duplicate_final_merged_files.csv"

# Backup si ya existía un consolidado anterior
if output_path.exists():
    backup_path = output_dir / "all_final_merged_segments_previous_backup.csv"
    shutil.copy2(output_path, backup_path)
    print("Backup creado:", backup_path)

# Guardamos consolidado corregido
df_segments.to_csv(output_path, index=False)
df_segments.to_csv(dedup_output_path, index=False)

# Guardamos auditorías
audit_files.to_csv(audit_path, index=False)
dropped_files.to_csv(dropped_path, index=False)

print("Archivo creado:", output_path)
print("Archivo duplicado de seguridad:", dedup_output_path)
print("Auditoría de archivos:", audit_path)
print("Archivos descartados por duplicidad:", dropped_path)

print("\nResumen final:")
print("Audios únicos por audio_file:", df_segments["audio_file"].nunique())
print("Audios únicos por audio_id_base:", df_segments["audio_id_base"].nunique())
print("Segmentos totales:", len(df_segments))

Backup creado: /home/jupyter/TFM_ProcesadoDeAudios/data/diarization_outputs/consolidated/all_final_merged_segments_previous_backup.csv
Archivo creado: /home/jupyter/TFM_ProcesadoDeAudios/data/diarization_outputs/consolidated/all_final_merged_segments.csv
Archivo duplicado de seguridad: /home/jupyter/TFM_ProcesadoDeAudios/data/diarization_outputs/consolidated/all_final_merged_segments_dedup.csv
Auditoría de archivos: /home/jupyter/TFM_ProcesadoDeAudios/data/diarization_outputs/consolidated/audit_final_merged_files.csv
Archivos descartados por duplicidad: /home/jupyter/TFM_ProcesadoDeAudios/data/diarization_outputs/consolidated/dropped_duplicate_final_merged_files.csv

Resumen final:
Audios únicos por audio_file: 1181
Audios únicos por audio_id_base: 1181
Segmentos totales: 40352
